# Serialización del mejor modelo y MLflow Model Registry

En este notebook serializamos el pipeline completo del mejor modelo
(**Exp 0: LogisticRegression + TF-IDF**, F1-macro test: 0.9053)
y lo registramos en el MLflow Model Registry.

Pasos:
1. Carga del mejor modelo
2. Serialización completa del pipeline con naming convention claro
3. Loguear el modelo en MLflow con `mlflow.sklearn.log_model`
4. Registrar en el Model Registry y transitar a stage `Production`
5. Verificación del registro

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_artificial"))
        break

import functions
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"

## 1. Carga del mejor modelo

In [16]:
import json
import joblib

with open("model/mejor_modelo_seleccion.json", encoding="utf-8") as f:
    seleccion = json.load(f)

modelo = joblib.load(seleccion["model_file"])
tfidf  = joblib.load(seleccion["tfidf_file"])

print(f"Modelo seleccionado: {seleccion['nombre']}")
print(f"Tipo: {seleccion['model_type']}")
print(f"Vocabulario TF-IDF: {len(tfidf.vocabulary_)} términos")
if hasattr(modelo, "classes_"):
    print(f"Clases: {sorted(modelo.classes_.tolist())}")
print("\nMétricas en test:")
print(f"  F1-macro:  {seleccion['test_f1_macro']}")
print(f"  Accuracy:  {seleccion['test_accuracy']}")
print(f"  ROC AUC:   {seleccion['test_roc_auc']}")

Modelo seleccionado: Exp 0: LogReg + TF-IDF
Tipo: LogisticRegression
Vocabulario TF-IDF: 3773 términos
Clases: ['alto_riesgo', 'inaceptable', 'riesgo_limitado', 'riesgo_minimo']

Métricas en test:
  F1-macro:  0.9053
  Accuracy:  0.9111
  ROC AUC:   0.9948


## 2. Serialización completa del pipeline

In [17]:
from functions import guardar_pipeline_completo

features_label = "tfidf_bigramas" if not seleccion["needs_manual_features"] else "tfidf_bigramas+manuales"

path_modelo, path_tfidf, path_meta = guardar_pipeline_completo(
    modelo=modelo,
    tfidf=tfidf,
    label_encoder=None,
    metadata={
        "experimento":   seleccion["experimento"],
        "nombre":        seleccion["nombre"],
        "test_f1_macro": seleccion["test_f1_macro"],
        "test_accuracy": seleccion["test_accuracy"],
        "test_roc_auc":  seleccion["test_roc_auc"],
        "features":      features_label,
        "tfidf_n_terms": len(tfidf.vocabulary_),
        "criterio":      seleccion["criterio"],
    },
    output_dir="model",
)

Modelo guardado:      model\mejor_modelo.joblib
Vectorizador guardado: model\mejor_modelo_tfidf.joblib
Metadata guardada:    model\model_metadata.json


## 3. Loguear el modelo en MLflow con sklearn.log_model

Usamos `mlflow.sklearn.log_model` con `registered_model_name` para que
MLflow cree automáticamente la primera versión en el Model Registry.

In [ ]:
import mlflow
import mlflow.sklearn
from functions import configure_mlflow, MLFLOW_EXPERIMENT

REGISTERED_NAME = "clasificador_riesgo_ia"

run_id = None

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="mejor_modelo_produccion") as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model_type":         seleccion["model_type"],
            "tfidf_max_features": 5000,
            "tfidf_ngram_range":  "(1, 2)",
            "tfidf_sublinear_tf": True,
            "features":           features_label,
            "experimento":        seleccion["experimento"],
        })

        mlflow.log_metrics({
            "test_f1_macro": seleccion["test_f1_macro"],
            "test_accuracy": seleccion["test_accuracy"],
            "test_roc_auc":  seleccion["test_roc_auc"],
        })

        mlflow.set_tags({
            "best_model": "true",
            "criterio":   seleccion["criterio"],
        })

        mlflow.log_artifact(path_modelo)
        mlflow.log_artifact(path_tfidf)
        mlflow.log_artifact(path_meta)

        mlflow.sklearn.log_model(
            sk_model=modelo,
            artifact_path="modelo",
        )

    print(f"✓ Run completado. Run ID: {run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

## 4. Transitar a stage Production en el Model Registry

In [ ]:
from functions import registrar_modelo_en_registry

if run_id is None:
    print("⚠ run_id no disponible — MLflow no conectó. Saltando registro en Model Registry.")
else:
    model_version = registrar_modelo_en_registry(
        run_id=run_id,
        artifact_path="modelo",
        registered_name=REGISTERED_NAME,
        stage="Production",
    )

## 5. Verificación del registro

In [20]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Listar todas las versiones del modelo
versiones = client.search_model_versions(f"name='{REGISTERED_NAME}'")
print(f"Versiones registradas de '{REGISTERED_NAME}':")
for v in versiones:
    print(f"  v{v.version} | stage={v.current_stage} | run_id={v.run_id[:8]}...")

# Verificar que hay una versión en Production
prod_versions = [v for v in versiones if v.current_stage == "Production"]
if prod_versions:
    print(f"\n✓ Modelo en Production: v{prod_versions[0].version}")
else:
    print("\n⚠ No hay versiones en Production")

## Resumen final

| Artefacto | Ruta |
|-----------|------|
| Modelo serializado | `model/mejor_modelo.joblib` |
| Vectorizador serializado | `model/mejor_modelo_tfidf.joblib` |
| Metadata | `model/model_metadata.json` |
| MLflow Registry | `clasificador_riesgo_ia` (stage: Production) |

In [21]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
# (El logging principal ya se hizo en la celda 3 con start_run.)
# Esta celda es un resumen de verificación final.
try:
    client = MlflowClient()
    versiones = client.search_model_versions(f"name='{REGISTERED_NAME}'")
    prod = [v for v in versiones if v.current_stage == "Production"]
    print(f"✓ '{REGISTERED_NAME}' — {len(versiones)} versión(es) registrada(s)")
    if prod:
        print(f"  En Production: v{prod[0].version} (run {prod[0].run_id[:8]}...)")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")